# Анализ · LLM размечают сказки по функциям Проппа

Только анализ, без запросов к моделям. Рядом с блокнотом должны лежать:
- папка `cache/` с файлами ответов: `propp_member1.jsonl`, `propp_member2.jsonl`, `propp_member3.jsonl`, `propp_yandex.jsonl`;
- `propp_gold.csv` — эталон/.

Результаты (таблицы и графики) сохраняются в папку `results/`.

## 1. Импорты и настройки

In [ ]:
%pip install -q pandas numpy scipy matplotlib krippendorff

In [ ]:
import re, json, glob, itertools, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from IPython.display import display
except ImportError:
    display = print
Path("results").mkdir(exist_ok=True)

MODELS = {   # три основные модели (Groq)
    "gpt-oss-20b": {}, "gpt-oss-120b": {}, "qwen3.8-27b": {},
}
BONUS_MODELS = {   # дополнительные (Yandex AI Studio)
    "yandexgpt-5.1": {}, "deepseek-v4.1-flash": {},
}
ALL_MODELS = {**MODELS, **BONUS_MODELS}
CONDITIONS    = ["A_names", "B_defs", "C_fewshot"]
FEWSHOT_TALE  = "64"           # пример в условии C → в сравнении условий не участвует
EXCLUDE_TALES = ["116"]        # не влезла в лимит Groq
TEMP_TALES    = ["72", "70", "67"]   # сказки проверки температуры (ключи с суффиксом _T03)

## 2. Эталон и функции

In [ ]:
def load_gold(path):
    g = pd.read_csv(path, dtype=str).set_index("tale_id")
    return g.drop(index=[t for t in EXCLUDE_TALES if t in g.index])

GOLD = load_gold("propp_gold.csv")

def gold_nums(tid, gold=None):
    gold = GOLD if gold is None else gold
    return [int(t) for t in gold.loc[tid, "gold_num"].split() if t.isdigit()]   # 'N' отбрасываем

FUNCS = {
    8:  ("A",  "Вредительство или недостача",
         "антагонист наносит вред кому-то из семьи (похищает, убивает, изгоняет, околдовывает, подменяет, грозит) "
         "ИЛИ кому-то из семьи чего-то не хватает, хочется что-то иметь (невеста, чудесный предмет, лекарство)"),
    9:  ("B",  "Посредничество",
         "о беде или недостаче сообщают; к герою обращаются с просьбой или приказом, его посылают или отпускают"),
    10: ("C",  "Начинающееся противодействие",
         "искатель соглашается или решается действовать против беды"),
    11: ("↑",  "Отправка", "герой покидает дом"),
    12: ("Д",  "Первая функция дарителя",
         "героя испытывают, выспрашивают, просят о помощи или нападают на него — это подготавливает получение "
         "волшебного средства или помощника"),
    13: ("Г",  "Реакция героя",
         "герой реагирует на действия будущего дарителя: выдерживает или не выдерживает испытание, оказывает "
         "услугу, щадит, отказывает"),
    14: ("Z",  "Получение волшебного средства",
         "в распоряжение героя попадает волшебное средство или помощник: дарят, находят, покупают, появляется "
         "сам, предлагает свои услуги"),
    15: ("R",  "Пространственное перемещение",
         "героя переносят, доставляют или приводят к месту, где находится предмет поисков"),
    16: ("Б",  "Борьба", "герой и антагонист вступают в непосредственную борьбу (бой, состязание)"),
    17: ("К",  "Клеймение", "героя метят: рана, кольцо, полотенце, поцелуй"),
    18: ("П",  "Победа", "антагонист побеждён: в бою, в состязании, убит, изгнан"),
    19: ("Л",  "Ликвидация беды или недостачи",
         "исходная беда или недостача устранена: похищенное добыто, убитый оживлён, заколдованный расколдован"),
    20: ("↓",  "Возвращение", "герой возвращается"),
    21: ("Пр", "Преследование", "героя преследуют: гонятся, летят вдогонку, пытаются убить"),
    22: ("Сп", "Спасение от преследования",
         "герой спасается: бегство, препятствия на пути погони, прятание, превращение"),
    23: ("°",  "Неузнанное прибытие", "герой неузнанным прибывает домой или в другую страну"),
    24: ("Ф",  "Притязания ложного героя", "ложный герой предъявляет необоснованные притязания"),
    25: ("З",  "Трудная задача", "герою предлагают трудную задачу"),
    26: ("Р",  "Решение задачи", "задача решена"),
    27: ("У",  "Узнавание", "героя узнают (по метке, кольцу, решённой задаче)"),
    28: ("О",  "Обличение", "ложный герой или антагонист обличён"),
    29: ("Т",  "Трансфигурация", "герою даётся новый облик"),
    30: ("Н",  "Наказание", "враг наказан"),
    31: ("С",  "Свадьба", "герой вступает в брак и/или воцаряется (или получает награду)"),
}
FUNC_IDS = sorted(FUNCS)
SYM = {k: v[0] for k, v in FUNCS.items()}


print("сказок в эталоне:", len(GOLD))
display(pd.DataFrame({"title": GOLD["title"], "ходов": GOLD["moves"],
                      "длина эталона": {t: len(gold_nums(t)) for t in GOLD.index}}))

## 3. Разбор ответов моделей

In [ ]:
ROMAN = {"VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12, "XIII": 13, "XIV": 14, "XV": 15, "XVI": 16,
         "XVII": 17, "XVIII": 18, "XIX": 19, "XX": 20, "XXI": 21, "XXII": 22, "XXIII": 23, "XXIV": 24,
         "XXV": 25, "XXVI": 26, "XXVII": 27, "XXVIII": 28, "XXIX": 29, "XXX": 30, "XXXI": 31}

def to_func(v):
    if isinstance(v, (int, float)) and int(v) in FUNCS:
        return int(v)
    s = str(v).strip()
    m = re.match(r"^\s*(\d{1,2})\b", s)
    if m and int(m.group(1)) in FUNCS:
        return int(m.group(1))
    if s.upper() in ROMAN:
        return ROMAN[s.upper()]
    return None

def count_invalid(text):
    # сколько элементов с номером вне 8–31 (модель выдумала функцию) — отбрасываются при разборе
    try:
        d = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return sum(1 for it in d.get("functions", []) if to_func(it.get("f") if isinstance(it, dict) else it) is None)
    except Exception:
        return 0

def parse_answer(text):
    # → список (номер, цитата) или None
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)
    a, b = text.find("{"), text.rfind("}")
    cands = [text[a:b + 1]] if 0 <= a < b else []
    a2, b2 = text.find("["), text.rfind("]")
    if 0 <= a2 < b2:
        cands.append(text[a2:b2 + 1])
    for c in cands:
        try:
            d = json.loads(c)
        except Exception:
            continue
        items = d.get("functions") if isinstance(d, dict) else d
        if not isinstance(items, list):
            continue
        out = []
        for it in items:
            if isinstance(it, dict):
                f = to_func(it.get("f", it.get("function", it.get("id"))))
                q = str(it.get("quote", ""))
            else:
                f, q = to_func(it), ""
            if f is not None:
                out.append((f, q))
        return out
    nums = re.findall(r'"f"\s*:\s*"?(\d{1,2})', text)   # JSON сломан (кавычки внутри цитат) — достаём номера
    if nums:
        return [(int(n), "") for n in nums if int(n) in FUNCS]
    return None


def load_cache():
    cache = {}
    for p in sorted(glob.glob("cache/*.jsonl")):
        for line in Path(p).read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                if rec.get("raw"):
                    cache[rec["key"]] = rec
    return cache

In [ ]:
CACHE = load_cache()
print("файлов в cache/:", sorted(glob.glob("cache/*.jsonl")))
rows = []
for key, rec in CACHE.items():
    model, cond, it, tid = key.split("|")
    if model not in ALL_MODELS or cond not in CONDITIONS or tid not in GOLD.index:
        continue
    parsed = parse_answer(rec["raw"])
    rows.append(dict(model=model, cond=cond, iter=int(it[2:]), tale=tid,
                     ok=parsed is not None,
                     empty=parsed == [],
                     n_invalid=count_invalid(rec["raw"]),
                     seq=[f for f, _ in parsed] if parsed else [],
                     quotes=[q for _, q in parsed] if parsed else [],
                     **{k: rec.get("usage", {}).get(k) for k in ("prompt_tokens", "cached_tokens", "completion_tokens")}))
RES = pd.DataFrame(rows)
print("ответов:", len(RES), "| не разобрано:", int((~RES["ok"]).sum()),
      "| пустой список функций:", int(RES["empty"].sum()),
      "| выдуманных номеров функций (вне 8–31):", int(RES["n_invalid"].sum()))
display(RES.groupby(["model", "cond"])[["empty", "n_invalid"]].sum())
display(RES.pivot_table(index="model", columns="cond", values="ok", aggfunc="count", fill_value=0))
RES.drop(columns=["quotes"]).to_csv("results/answers_long.csv", index=False)

## 4. Метрики

Перед сравнением и эталон, и ответ модели **схлопываем подряд идущие одинаковые функции** (Б Б → Б), чтобы не наказывать за то, что модель разбила один эпизод на два предложения.

1. **F1 по набору функций** (мультимножество): сколько функций совпало без учёта порядка.
2. **Сходство последовательностей** = 1 − расстояние Левенштейна / длина более длинной. Учитывает порядок.
3. **Консистентность**: те же две метрики попарно между 5 итерациями одной модели (10 пар), среднее.
4. **Альфа Криппендорфа** (номинальная): единица = (сказка, функция), значение = есть функция в разметке или нет (по большинству из 5 итераций). Между моделями — отдельно для каждого условия. Внутри модели — итерации как разные разметчики.

In [ ]:
def collapse(seq):
    return [f for i, f in enumerate(seq) if i == 0 or f != seq[i - 1]]

def multiset_prf(pred, gold):
    p, g = Counter(pred), Counter(gold)
    tp = sum((p & g).values())
    prec = tp / max(sum(p.values()), 1)
    rec = tp / max(sum(g.values()), 1)
    f1 = 0.0 if tp == 0 else 2 * prec * rec / (prec + rec)
    return prec, rec, f1

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def seq_sim(a, b):
    if not a and not b: return 1.0
    return 1 - levenshtein(a, b) / max(len(a), len(b))

GOLD_SEQ = {t: collapse(gold_nums(t)) for t in GOLD.index}
PRESENT = [m for m in ALL_MODELS if m in set(RES.loc[RES["ok"], "model"])]   # модели, по которым есть ответы
print("модели в анализе:", PRESENT)

ok = RES[RES["ok"]].copy()
ok["seq_c"] = ok["seq"].apply(collapse)
m = ok.apply(lambda r: multiset_prf(r["seq_c"], GOLD_SEQ[r["tale"]]), axis=1, result_type="expand")
ok[["precision", "recall", "f1"]] = m
ok["seq_sim"] = ok.apply(lambda r: seq_sim(r["seq_c"], GOLD_SEQ[r["tale"]]), axis=1)
ok["f1_set"] = ok.apply(lambda r: multiset_prf(sorted(set(r["seq_c"])), sorted(set(GOLD_SEQ[r["tale"]])))[2], axis=1)
ok["completion_tokens"] = pd.to_numeric(ok["completion_tokens"], errors="coerce")
ok["len_pred"] = ok["seq_c"].apply(len)
ok["len_gold"] = ok["tale"].map(lambda t: len(GOLD_SEQ[t]))
ok.drop(columns=["quotes"]).to_csv("results/scores_long.csv", index=False)

# средние по сказке (5 итераций) → потом среднее и SD по сказкам
per_tale_all = ok.groupby(["model", "cond", "tale"])[["precision", "recall", "f1", "f1_set", "seq_sim", "len_pred", "completion_tokens"]].mean().reset_index()
# условия сравниваем на ОБЩИХ сказках: без примера для few-shot
COMMON = [t for t in GOLD.index if t != FEWSHOT_TALE]
per_tale = per_tale_all[per_tale_all["tale"].isin(COMMON)]
ACC = per_tale.groupby(["model", "cond"]).agg(
    F1_mean=("f1", "mean"), F1_sd=("f1", "std"),
    F1set_mean=("f1_set", "mean"), F1set_sd=("f1_set", "std"),
    tokens_out=("completion_tokens", "mean"),
    SeqSim_mean=("seq_sim", "mean"), SeqSim_sd=("seq_sim", "std"),
    Precision=("precision", "mean"), Recall=("recall", "mean"),
    len_pred=("len_pred", "mean"), n_tales=("tale", "nunique"),
).round(3)
print(f"ТОЧНОСТЬ относительно эталона Проппа (среднее и SD по {len(COMMON)} общим сказкам, без {FEWSHOT_TALE})")
display(ACC)
ACC.to_csv("results/accuracy.csv")

In [ ]:
# Сравнение моделей на ОДИНАКОВЫХ сказках (защита от разного покрытия)
need = set(PRESENT)
full = per_tale.groupby(["tale", "cond"])["model"].apply(lambda s: need <= set(s))
fair_keys = full[full].reset_index()[["tale", "cond"]]
FAIR = per_tale.merge(fair_keys).groupby(["model", "cond"])[["f1", "f1_set", "seq_sim"]].mean().round(3)
print("Сказки, где есть ответы всех моделей:", fair_keys.groupby("cond")["tale"].apply(list).to_dict())
display(FAIR)
FAIR.to_csv("results/accuracy_fair.csv")

In [ ]:
# Консистентность: попарно между итерациями одной модели на одной сказке
cons_rows = []
for (model, cond, tid), g in ok.groupby(["model", "cond", "tale"]):
    seqs = list(g.sort_values("iter")["seq_c"])
    if len(seqs) < 2: continue
    f1s, sims = [], []
    for a, b in itertools.combinations(seqs, 2):
        f1s.append(multiset_prf(a, b)[2]); sims.append(seq_sim(a, b))
    cons_rows.append(dict(model=model, cond=cond, tale=tid, n_iter=len(seqs),
                          cons_f1=np.mean(f1s), cons_seq=np.mean(sims),
                          identical=float(all(s == seqs[0] for s in seqs))))
CONS_T = pd.DataFrame(cons_rows)
CONS = CONS_T.groupby(["model", "cond"]).agg(
    ConsF1_mean=("cons_f1", "mean"), ConsF1_sd=("cons_f1", "std"),
    ConsSeq_mean=("cons_seq", "mean"), ConsSeq_sd=("cons_seq", "std"),
    share_identical=("identical", "mean"),
).round(3)
print("КОНСИСТЕНТНОСТЬ между итерациями (1 = все 5 разметок совпадают)")
display(CONS)
CONS.to_csv("results/consistency.csv")

In [ ]:
try:
    import krippendorff
except ImportError:
    krippendorff = None

def alpha_nominal(M):
    # своя реализация номинальной альфы (матрица совпадений); на примере из Krippendorff 2011 даёт 0.743
    vals = sorted(set(M[~np.isnan(M)].tolist()))
    idx = {v: i for i, v in enumerate(vals)}
    o = np.zeros((len(vals), len(vals)))
    for u in range(M.shape[1]):
        col = M[:, u]; col = col[~np.isnan(col)]
        if len(col) < 2: continue
        c = Counter(col.tolist())
        for a in c:
            for b in c:
                o[idx[a], idx[b]] += c[a] * (c[b] - (a == b)) / (len(col) - 1)
    n_c = o.sum(1); n = n_c.sum()
    De = (n * n - (n_c ** 2).sum()) / (n - 1)
    return 1 - (n - np.trace(o)) / De if De else np.nan

def presence_matrix(df, coder_col):
    # строки = разметчики, столбцы = (сказка, функция), значение 0/1
    coders = sorted(df[coder_col].unique())
    units = [(t, f) for t in sorted(df["tale"].unique()) for f in FUNC_IDS]
    M = np.full((len(coders), len(units)), np.nan)
    for i, c in enumerate(coders):
        sub = df[df[coder_col] == c].set_index("tale")["seq_c"]
        for j, (t, f) in enumerate(units):
            if t in sub.index:
                M[i, j] = float(f in sub[t])
    return coders, M

def kalpha(M):
    M = M[:, ~np.all(np.isnan(M), axis=0)]
    if M.shape[0] < 2 or np.nanstd(M) == 0: return np.nan
    if krippendorff is not None:
        return krippendorff.alpha(reliability_data=M, level_of_measurement="nominal")
    return alpha_nominal(M)

# большинство из 5 итераций: функция есть, если она есть в ≥ половине итераций
def majority(g):
    n = len(g)
    cnt = Counter(f for s in g for f in set(s))
    return sorted(f for f, c in cnt.items() if c >= n / 2)

MAJ = ok.groupby(["model", "cond", "tale"])["seq_c"].apply(majority).reset_index()

alpha_rows = []
for cond in CONDITIONS:
    sub_all = MAJ[MAJ["cond"] == cond]
    sub = sub_all[sub_all["model"].isin(MODELS)]          # согласованность считаем по 3 основным моделям
    _, M = presence_matrix(sub, "model")
    alpha_rows.append(dict(level="между моделями", cond=cond, model="3 основные", alpha=kalpha(M)))
    if sub_all["model"].nunique() > sub["model"].nunique():
        _, Mb = presence_matrix(sub_all, "model")
        alpha_rows.append(dict(level="между моделями", cond=cond, model="+ бонусные", alpha=kalpha(Mb)))
    # + эталон как ещё один «разметчик» — насколько модели вместе с Проппом согласованы
    gold_df = pd.DataFrame([dict(model="PROPP", tale=t, seq_c=GOLD_SEQ[t]) for t in sub["tale"].unique()])
    _, M2 = presence_matrix(pd.concat([sub, gold_df]), "model")
    alpha_rows.append(dict(level="модели + Пропп", cond=cond, model="3 основные", alpha=kalpha(M2)))
    for model in PRESENT:
        s = ok[(ok["cond"] == cond) & (ok["model"] == model)]
        if s.empty: continue
        _, M3 = presence_matrix(s, "iter")
        alpha_rows.append(dict(level="внутри модели (итерации)", cond=cond, model=model, alpha=kalpha(M3)))
ALPHA = pd.DataFrame(alpha_rows).round(3)
display(ALPHA)
ALPHA.to_csv("results/krippendorff.csv", index=False)

In [ ]:
# Помогают ли определения и пример? Парный тест Уилкоксона по общим сказкам, для каждой модели
PAIRS = [("A_names", "B_defs"), ("B_defs", "C_fewshot"), ("A_names", "C_fewshot")]
test_rows = []
for model in PRESENT:
    for metric in ["f1", "seq_sim"]:
        w = per_tale[per_tale["model"] == model].pivot(index="tale", columns="cond", values=metric)
        for c1, c2 in PAIRS:
            if c1 not in w or c2 not in w: continue
            ww = w[[c1, c2]].dropna()
            if len(ww) < 5: continue
            diff = ww[c2] - ww[c1]
            p = stats.wilcoxon(ww[c2], ww[c1]).pvalue if (diff != 0).any() else 1.0
            test_rows.append(dict(model=model, metric=metric, compare=f"{c2} − {c1}", n=len(ww),
                                  mean_1=ww[c1].mean(), mean_2=ww[c2].mean(), diff=diff.mean(), p_wilcoxon=p))
TESTS = pd.DataFrame(test_rows).round(3)
display(TESTS)
TESTS.to_csv("results/conditions_test.csv", index=False)

In [ ]:
# Объединённый тест по всем моделям: пары (модель, сказка)
w = per_tale.pivot_table(index=["model", "tale"], columns="cond", values="f1")
pooled = []
for c1, c2 in [("A_names", "B_defs"), ("B_defs", "C_fewshot"), ("A_names", "C_fewshot")]:
    x = w[[c1, c2]].dropna()
    pooled.append(dict(compare=f"{c2} − {c1}", n_pairs=len(x), diff=(x[c2] - x[c1]).mean(),
                       share_better=((x[c2] - x[c1]) > 0).mean(), p_wilcoxon=stats.wilcoxon(x[c2], x[c1]).pvalue))
POOLED = pd.DataFrame(pooled).round(4)
display(POOLED)
POOLED.to_csv("results/conditions_test_pooled.csv", index=False)

# Различаются ли модели? Тест Фридмана по сказкам (условие B_defs)
wb = per_tale[per_tale.cond == "B_defs"].pivot(index="tale", columns="model", values="f1").dropna()
print("F1 (B_defs) по моделям:"); display(wb.mean().sort_values(ascending=False).round(3))
print("Фридман: p = %.4g" % stats.friedmanchisquare(*[wb[c] for c in wb]).pvalue)
print("\nТрудность сказок (средний F1 по моделям, B_defs):")
display(per_tale[per_tale.cond == "B_defs"].groupby("tale")["f1"].mean().round(2).sort_values())

## 5. Разбор ошибок по функциям

Для каждой функции: как часто модель её **находит**, когда она есть у Проппа (recall), и как часто ставит её **там, где у Проппа её нет** (лишние). Считаем по наличию функции в сказке.

In [ ]:
fn_rows = []
for _, r in ok.iterrows():
    g, p = set(GOLD_SEQ[r["tale"]]), set(r["seq_c"])
    for f in FUNC_IDS:
        fn_rows.append(dict(model=r["model"], cond=r["cond"], f=f, in_gold=f in g, in_pred=f in p))
FN = pd.DataFrame(fn_rows)

def fn_stats(d):
    tp = (d.in_gold & d.in_pred).sum(); fn = (d.in_gold & ~d.in_pred).sum(); fp = (~d.in_gold & d.in_pred).sum()
    return pd.Series(dict(gold_cnt=int(d.in_gold.sum()),
                          recall=tp / (tp + fn) if tp + fn else np.nan,
                          extra=int(fp)))

BYF = FN.groupby(["cond", "f"])[["in_gold", "in_pred"]].apply(fn_stats).reset_index()
BYF["функция"] = BYF["f"].map(lambda f: f"{f} {SYM[f]} {FUNCS[f][1]}")
display(BYF.sort_values("f").pivot(index="f", columns="cond", values=["recall", "extra"])
        .rename(index=lambda f: f"{f} {SYM[f]} {FUNCS[f][1]}").round(2))
BYF.to_csv("results/by_function.csv", index=False)

In [ ]:
# Рядом: эталон и разметка модели (с цитатами) — для качественного анализа
def show(tid, model, cond, it=1):
    r = ok[(ok.tale == tid) & (ok.model == model) & (ok.cond == cond) & (ok.iter == it)]
    print(f"Сказка {tid} «{GOLD.loc[tid, 'title']}» | {model} | {cond} | итерация {it}")
    print("Пропп :", " ".join(SYM[f] for f in GOLD_SEQ[tid]))
    if r.empty:
        print("нет ответа"); return
    r = r.iloc[0]
    print("модель:", " ".join(SYM[f] for f in r["seq_c"]))
    print()
    for f, q in zip(r["seq"], r["quotes"]):
        print(f"  {f:>2} {SYM[f]:<2} {FUNCS[f][1]:<32} «{q}»")

show("64", "gpt-oss-120b", "B_defs")

## 6. Графики

In [ ]:
order = PRESENT
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (tbl, col, sd, title) in zip(axes, [
        (ACC, "F1_mean", "F1_sd", "Точность: F1 (с учётом повторов)"),
        (ACC, "SeqSim_mean", "SeqSim_sd", "Точность: сходство последовательностей"),
        (CONS, "ConsF1_mean", "ConsF1_sd", "Консистентность между итерациями (F1)")]):
    t = tbl.reset_index()
    x = np.arange(len(order)); w = 0.8 / len(CONDITIONS)
    for i, cond in enumerate(CONDITIONS):
        s = t[t["cond"] == cond].set_index("model").reindex(order)
        ax.bar(x + (i - (len(CONDITIONS) - 1) / 2) * w, s[col], w, yerr=s[sd], capsize=3, label=cond)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=20); ax.set_ylim(0, 1.05); ax.set_title(title)
axes[0].legend()
plt.tight_layout(); plt.savefig("results/fig_accuracy_consistency.png", dpi=160); plt.show()

In [ ]:
# Падает ли точность с длиной эталона (числом функций / ходов)?
fig, ax = plt.subplots(figsize=(7, 4.5))
pt = per_tale.merge(GOLD[["moves"]].astype(int), left_on="tale", right_index=True)
pt["len_gold"] = pt["tale"].map(lambda t: len(GOLD_SEQ[t]))
for model in PRESENT:
    s = pt[(pt.model == model) & (pt.cond == "B_defs")]
    ax.scatter(s["len_gold"], s["f1"], label=model, alpha=.8)
ax.set_xlabel("длина эталона (функций)"); ax.set_ylabel("F1 (B_defs)"); ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig("results/fig_length.png", dpi=160); plt.show()
rho = stats.spearmanr(pt["len_gold"], pt["f1"])
print(f"Спирмен длина эталона ~ F1 (все модели и условия): rho={rho.statistic:.2f}, p={rho.pvalue:.3f}")

In [ ]:
# Recall по функциям (условие B_defs), по моделям
H = FN[FN.cond == "B_defs"].groupby(["model", "f"])[["in_gold", "in_pred"]].apply(
    lambda d: (d.in_gold & d.in_pred).sum() / max(d.in_gold.sum(), 1) if d.in_gold.sum() else np.nan).unstack()
H = H.reindex(order)
H = H.loc[:, H.notna().any()]
fig, ax = plt.subplots(figsize=(14, 3.2))
im = ax.imshow(H.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_yticks(range(len(H.index))); ax.set_yticklabels(H.index)
ax.set_xticks(range(len(H.columns))); ax.set_xticklabels([SYM[f] for f in H.columns])
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        if not np.isnan(v): ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="w" if v < .5 else "k")
ax.set_title("Recall по функциям (B_defs): доля случаев, когда функция Проппа найдена")
plt.colorbar(im, ax=ax, fraction=.02)
plt.tight_layout(); plt.savefig("results/fig_by_function.png", dpi=160); plt.show()

## 7. Проверка температуры (T = 0.7 и T = 0.3)

Те же 3 сказки, все три условия, 5 итераций. Ответы на 0.3 лежат в кеше с суффиксом `_T03`.

In [ ]:
trow = []
for key, rec in CACHE.items():
    model, cond, it, tid = key.split("|")
    base = cond.replace("_T03", "")
    if tid in TEMP_TALES and base in CONDITIONS and model in ALL_MODELS:
        p = parse_answer(rec["raw"])
        if p is None: continue
        s = collapse([f for f, _ in p])
        trow.append(dict(model=model, cond=base, T=0.3 if cond.endswith("_T03") else 0.7, tale=tid, seq=s,
                         f1=multiset_prf(s, GOLD_SEQ[tid])[2],
                         f1_set=multiset_prf(sorted(set(s)), sorted(set(GOLD_SEQ[tid])))[2],
                         seq_sim=seq_sim(s, GOLD_SEQ[tid])))
TT = pd.DataFrame(trow)
cons = []
for (model, cond, T, tid), g in TT.groupby(["model", "cond", "T", "tale"]):
    pairs = list(itertools.combinations(list(g["seq"]), 2))
    if pairs:
        cons.append(dict(model=model, cond=cond, T=T, tale=tid, cons_f1=np.mean([multiset_prf(a, b)[2] for a, b in pairs])))
CONS_TT = pd.DataFrame(cons)
TEMP_RES = (TT.groupby(["model", "T"])[["f1", "f1_set", "seq_sim"]].mean()
            .join(CONS_TT.groupby(["model", "T"])[["cons_f1"]].mean()).round(3).unstack("T"))
print("По моделям:"); display(TEMP_RES)
TEMP_COND = (TT.groupby(["cond", "T"])[["f1"]].mean()
             .join(CONS_TT.groupby(["cond", "T"])[["cons_f1"]].mean()).round(3).unstack("T"))
print("По условиям:"); display(TEMP_COND)
TEMP_RES.to_csv("results/temperature_check.csv")